In [1]:
import os, rasterio, sys, rioxarray, pyflwdir, shutil, cdsapi, dotenv, calendar, glob, gc
sys.path.append('backend/app/')
from rasterio.features import rasterize
from dateutil.relativedelta import relativedelta
from rasterio.mask import mask
from datetime import datetime, timedelta
from netCDF4 import Dataset, date2num
import geopandas as gpd, pandas as pd
import numpy as np, xarray as xr
from backend.app.services import flow_functions
from shapely.geometry import Polygon, MultiPolygon
from shapely import force_2d
from pyflwdir import dem
from scipy.ndimage import sobel
from hydromt_wflow import WflowSbmModel
from tqdm import tqdm
from pathlib import Path
np.random.seed(42)

In [2]:
def keep_polygon(geom):
    if geom.geom_type == 'GeometryCollection':
        polys = [g for g in geom.geoms if isinstance(g, (Polygon, MultiPolygon))]
        if len(polys) == 0: return None
        return polys[0]
    return geom

def fix_invalid_polygon(gdf, cols):
    gdf_new, name = gdf.copy(), cols[0]
    gdf_valid, gdf_nan = gdf_new[gdf_new[name] != ''], gdf_new[gdf_new[name] == '']
    if gdf_nan.shape[0] > 0:
        gdf_valid['geometry'] = gdf_valid['geometry'].apply(keep_polygon)
        gdf_nan['geometry'] = gdf_nan['geometry'].apply(keep_polygon)
        # Spatial join nearest
        gdf_filled = gpd.sjoin_nearest(
            gdf_nan, gdf_valid[['geometry', name]], how='left', distance_col='dist'
        )
        gdf_filled = gdf_filled.drop_duplicates(subset='_id')
        gdf_new.loc[gdf_filled.index, cols] = gdf_valid.loc[gdf_filled['index_right'], cols].values
    gdf_new['geometry'] = gdf_new['geometry'].apply(keep_polygon)
    return gdf_new

def clip_catchment(catchment, terrain):
    clipped = terrain.rio.clip(catchment.geometry, catchment.crs, drop=False)
    clipped = clipped.fillna(-9999)
    clipped.rio.write_nodata(-9999, inplace=True)
    return clipped

def write_tif(path, terrain, geo, col=''):
    transform = terrain.rio.transform()
    if col == '': shapes = ((geom, 1) for geom in geo.geometry)
    else: shapes = ((geom, value) for geom, value in zip(geo.geometry, geo[col]))
    raster = rasterize(
        shapes=shapes, out_shape=(terrain.rio.height, terrain.rio.width),
        transform=transform, fill=-9999, dtype="float32", all_touched=True
    )
    meta = {
        "driver": "GTiff", "height": terrain.rio.height,
        "width": terrain.rio.width, "count": 1, "dtype": "float32", 
        "crs": terrain.rio.crs, "transform": transform, "nodata": -9999
    }
    with rasterio.open(path, "w", **meta) as dst:
        dst.write(raster, 1)

def write_geotiff(data, profile, output_path):
    profile.update(dtype=data.dtype, count=1, compress='lzw')
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(data, 1)

def is_valid_netcdf(path):
    try:
        with Dataset(path, "r") as ds:
           if len(ds.variables) == 0: return False
        return True
    except Exception:
        return False

def create_forcing(time, ny, nx, values, single_value=True):
    if single_value:
        data = np.empty((len(time), ny, nx), dtype=np.float32)
        data[:] = values[:, None, None]
    

    return data

In [3]:
sample_folder, test_folder = 'inputs', 'test'
catchment_path = os.path.join(sample_folder, 'catchment.geojson')
terrain_path = os.path.join(sample_folder, 'dtm10.tif')
soil_path = os.path.join(sample_folder, 'soil.geojson')
land_path = os.path.join(sample_folder, 'land.geojson')
river_path = os.path.join(sample_folder, 'river.geojson')
terrain = rioxarray.open_rasterio(terrain_path).squeeze()
catchment = gpd.read_file(catchment_path)
initial_param = [0.25,0.3,50,0,0,500,0,0,100]
soil = gpd.read_file(soil_path)
land = gpd.read_file(land_path)
river = gpd.read_file(river_path)

## Process catchment

In [10]:
catchment_UTM = catchment.to_crs(terrain.rio.crs)
catchment_dir = os.path.normpath(os.path.join(test_folder, 'data/catchment'))
if not os.path.exists(catchment_dir): os.makedirs(catchment_dir)
catchment_UTM.to_file(os.path.join(catchment_dir, 'catchment.gpkg'), driver='GPKG')

## Clip dtm to catchment

In [ ]:
terrain_clipped = clip_catchment(catchment_UTM, terrain)
terrain_out_path = os.path.normpath(os.path.join(f'{test_folder}/data/dtm', "dtm_clipped.tif"))
terrain_clipped.rio.to_raster(terrain_out_path)

## Create merit hydro data

In [ ]:
with rasterio.open(terrain_path) as src:
    dem_array = src.read(1).astype(np.float32)
    profile = src.profile
    transform = src.transform
    crs = src.crs
NODATA_DEM, NODATA_INT = -9999.0, 0
profile.update(dtype=np.float32, nodata=NODATA_DEM)
# Fill depressions
elevtn_array, flwdir_array = dem.fill_depressions(
    elevtn=dem_array, nodata=NODATA_DEM, max_depth=-1
)
elevtn_array = np.where(np.isfinite(elevtn_array), elevtn_array, NODATA_DEM)
# Create flow direction
flw = pyflwdir.from_array(
    data=flwdir_array, ftype='d8', transform=transform, 
    latlon=crs.is_geographic
)
# Create slope
dx, dy = transform.a, abs(transform.e)
# Gradient elevation
dzdx = sobel(elevtn_array, axis=1, mode='nearest') / (8 * dx)
dzdy = sobel(elevtn_array, axis=0, mode='nearest') / (8 * dy)
slope_array = np.sqrt(dzdx**2 + dzdy**2)
slope_array = np.where(elevtn_array == NODATA_DEM, NODATA_DEM, slope_array).astype(np.float32)
# Create basins
basins_array = flw.basins()
# Create stream order
uparea_array = flw.upstream_area(unit='km2')
# Create stream mask and stream order
stream_mask = uparea_array > 30
strord_array = flw.stream_order(type='strahler', mask=stream_mask)
merit_dir = os.path.normpath(f'{test_folder}/data/merit_hydro')
if not os.path.exists(merit_dir): os.makedirs(merit_dir)
write_geotiff(elevtn_array, profile, os.path.join(merit_dir, 'elevtn.tif'))
profile_flwdir = {**profile, 'dtype': np.uint8, 'nodata': NODATA_INT}
write_geotiff(flwdir_array, profile_flwdir, os.path.join(merit_dir, 'flwdir.tif'))
profile_slope = {**profile, 'dtype': np.float32, 'nodata': NODATA_DEM}
write_geotiff(slope_array, profile_slope, os.path.join(merit_dir, 'lndslp.tif'))
profile_basins = {**profile, 'dtype': np.int32, 'nodata': NODATA_INT}
write_geotiff(basins_array, profile_basins, os.path.join(merit_dir, 'basins.tif'))
profile_uparea = {**profile, 'dtype': np.float32, 'nodata': NODATA_DEM}
write_geotiff(uparea_array, profile_uparea, os.path.join(merit_dir, 'uparea.tif'))
profile_strord = {**profile, 'dtype': np.int16, 'nodata': NODATA_INT}
write_geotiff(strord_array, profile_strord, os.path.join(merit_dir, 'strord.tif'))

## Prepare forcing data from the customized area

In [4]:
weather_path = os.path.join(sample_folder, 'alesund_weather.csv')
weather = pd.read_csv(weather_path, parse_dates=['datetime'], index_col='datetime')
weather_new = weather.loc['2025-01-01 00:00:00':'2025-01-10 00:00:00']

In [27]:
time, crs = weather_new.index.to_numpy(), terrain.rio.crs
if crs is None: raise ValueError("Terrain has no crs")
ny, nx = terrain.rio.height, terrain.rio.width
forcing = {
    'precip': ['precip_mm', '(mm/h)'], 'temp': ['temp_C', '(degC)'],
    'kin': ['shortwave_Wm2', '(W/m^2)'], 'kout': ['longwave_Wm2', '(W/m^2)'],
    'wind': ['wind_mps', '(m/s)'], 'press_msl': ['pressure', '(Pa)']
}
forcing_dir = os.path.join(test_folder, 'data/forcing')
if not os.path.exists(forcing_dir): os.makedirs(forcing_dir)
out_path, datasets = os.path.join(forcing_dir, "my_forcing.nc"), {}
for item, values in forcing.items():
    data = weather_new[values[0]].values
    data_3d = create_forcing(time, ny, nx, data)
    datasets[item] = (('time', 'y', 'x'), data_3d, {'units': values[1]})
ds_final = xr.Dataset(
    data_vars=datasets, coords={"time": time, "y": terrain.y, "x": terrain.x}
)
ds_final.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
ds_final.rio.write_crs(crs, inplace=True)
encoding = {
    var: {"zlib": True, "complevel": 4, "shuffle": True, "chunksizes": (1, 256, 256)}
    for var in ds_final.data_vars
}
ds_final.to_netcdf(out_path, engine='netcdf4', encoding=encoding)

## Process river data

In [49]:
# Create a random value for each river
river_UTM = river.to_crs(terrain.rio.crs)
cols = {'width': (0.05, 2), 'depth': (1, 5), 'manning_n': (0.03, 0.06)}
river_cols = river_UTM.columns.drop('geometry', errors='ignore')
for col in river_cols:
    river_UTM[col] = pd.to_numeric(river_UTM[col], errors='coerce')
for col, (low, high) in cols.items():
    river_mask = river_UTM[col].isna() | (river_UTM[col] == 'None')
    river_UTM.loc[river_mask, col] = np.round(np.random.uniform(low, high, river_mask.sum()), 3)
river_UTM = river_UTM[river_UTM.is_valid].reset_index(drop=True)

In [54]:
# Process river
river_UTM["geometry"] = river_UTM.geometry.apply(lambda g: force_2d(g))
river_UTM = river_UTM.rename(columns={'width': 'rivwth', 'depth': 'rivdph'})
river_UTM = river_UTM[['rivwth', 'rivdph', 'manning_n', 'geometry']]
river_UTM.to_file(os.path.normpath(os.path.join(f'{test_folder}/data/river', 'river.gpkg')), driver='GPKG')

## Process soil data

In [24]:
# Fix invalid soil polygon
soil_UTM = soil.to_crs(terrain.rio.crs)
soil_cols = ['soil', 'theta_s', 'theta_r', 'k_sat_ver', 'soil_depth', 'conductivity_decay', 'brooks_corey']
soil_UTM = fix_invalid_polygon(soil_UTM, soil_cols)
soil_layers = ['theta_s', 'theta_r', 'k_sat_ver', 'soil_depth', 'conductivity_decay', 'brooks_corey']
soil_UTM = soil_UTM[soil_layers + ['geometry']]
soil_UTM = soil_UTM.rename(columns={
    'theta_s': 'thetaS', 'theta_r': 'thetaR', 'k_sat_ver': 'KsatVer', 'soil_depth': 'SoilThickness', 
    'conductivity_decay': 'f', 'brooks_corey': 'brooks_corey'
})
# for value in soil_layers:
#     soil_path = os.path.normpath(os.path.join('test/data/soil', f'{value}.tif'))
#     write_tif(soil_path, terrain, soil_UTM, value)

In [25]:
soil_UTM

,thetaS,thetaR,KsatVer,SoilThickness,f,brooks_corey,geometry
0,1.00,1.00,10000,0,0.000,0,"POLYGON ((57720.518 6953831.515, 57735.027 695..."
1,0.80,0.20,50,2500,0.008,4,"POLYGON ((56460.162 6954272.557, 56466.979 695..."
2,0.05,0.01,100,100,0.040,1,"POLYGON ((56737.154 6954525.509, 56735.95 6954..."
3,0.05,0.01,100,100,0.040,1,"POLYGON ((56611.11 6955086.338, 56608.731 6955..."
4,0.05,0.01,100,100,0.040,1,"POLYGON ((56320.572 6954681.186, 56319.367 695..."
...,...,...,...,...,...,...,...
1020,0.45,0.06,200,1800,0.015,5,"POLYGON ((66159.324 6957342.995, 66158.159 695..."
1021,0.80,0.20,50,2500,0.008,4,"POLYGON ((67067.687 6957450.225, 67066.51 6957..."
1022,0.80,0.20,50,2500,0.008,4,"POLYGON ((67195.962 6957424.265, 67194.799 695..."
1023,0.80,0.20,50,2500,0.008,4,"POLYGON ((67235.856 6957464.304, 67234.693 695..."


In [4]:
with rasterio.open(r"test\data\merit_hydro\elevtn.tif") as src:
    dem_array = src.read(1).astype(np.float32)
    profile = src.profile
    transform = src.transform
    crs = src.crs

In [7]:
crs

CRS.from_wkt('PROJCS["ETRS89 / UTM zone 33N",GEOGCS["ETRS89",DATUM["European_Terrestrial_Reference_System_1989",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6258"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4258"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",15],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","25833"]]')

In [15]:
dt = xr.open_zarr(r"C:\Users\vanln\.hydromt\artifact_data\latest\era5_hourly_zarr.zarr")
dt.close()

In [16]:
dt

<xarray.Dataset> Size: 567kB
Dimensions:      (time: 336, latitude: 7, longitude: 6)
Coordinates:
  * time         (time) datetime64[ns] 3kB 2010-02-01 ... 2010-02-14T23:00:00
  * latitude     (latitude) float32 28B 46.75 46.5 46.25 46.0 45.75 45.5 45.25
  * longitude    (longitude) float32 24B 11.75 12.0 12.25 12.5 12.75 13.0
Data variables:
    cape         (time, latitude, longitude) float32 56kB dask.array<chunksize=(336, 7, 6), meta=np.ndarray>
    d2m          (time, latitude, longitude) float32 56kB dask.array<chunksize=(336, 7, 6), meta=np.ndarray>
    kin          (time, latitude, longitude) float32 56kB dask.array<chunksize=(336, 7, 6), meta=np.ndarray>
    kout         (time, latitude, longitude) float32 56kB dask.array<chunksize=(336, 7, 6), meta=np.ndarray>
    precip       (time, latitude, longitude) float32 56kB dask.array<chunksize=(336, 7, 6), meta=np.ndarray>
    press_msl    (time, latitude, longitude) float32 56kB dask.array<chunksize=(336, 7, 6), meta=np.ndarray>
    spatial_ref  int32 4B ...
    tcwv         (time, latitude, longitude) float32 56kB dask.array<chunksize=(336, 7, 6), meta=np.ndarray>
    temp         (time, latitude, longitude) float32 56kB dask.array<chunksize=(336, 7, 6), meta=np.ndarray>
    u10          (time, latitude, longitude) float32 56kB dask.array<chunksize=(336, 7, 6), meta=np.ndarray>
    v10          (time, latitude, longitude) float32 56kB dask.array<chunksize=(336, 7, 6), meta=np.ndarray>
Attributes:
    category:        meteo
    history:         Extracted from Copernicus Climate Data Store
    paper_doi:       10.1002/qj.3803
    paper_ref:       Hersbach et al. (2019)
    source_license:  https://cds.climate.copernicus.eu/cdsapp/#!/terms/licenc...
    source_url:      https://doi.org/10.24381/cds.bd0915c6
    source_version:  ERA5 hourly data on pressure levels

In [8]:
# Fix invalid land polygon
land_UTM = land.to_crs(terrain.rio.crs)
land_cols = ['land', 'LAI', 'root_depth', 'interception', 'manning_n', 'albedo', 'kc']
land_UTM = fix_invalid_polygon(land_UTM, land_cols)
land_layers = ['LAI', 'root_depth', 'interception', 'manning_n', 'albedo', 'kc']
for value in land_layers:
    land_path = os.path.normpath(os.path.join('test/data/landuse', f'{value}.tif'))
    write_tif(land_path, terrain, land_UTM, value)

In [30]:
# Create raster and lookup table (used for calibration)
land_class = land.to_crs(terrain.rio.crs).copy()
land_class['class'] = None
columns, table = np.unique(land_class['land'].values), {}
for id, item in enumerate(columns):
    temp = land_class[land_class['land'] == item]
    table[item] = np.float32(temp.iloc[0][land_layers].values)
    land_class.loc[temp.index, 'class'] = id
land_class = land_class[['class', 'geometry']]
land_class_path = os.path.normpath(os.path.join('test/data/lookup', 'land_classes.tif'))
write_tif(land_class_path, terrain, land_class, 'class')
# Create lookup table
lookup = pd.DataFrame.from_dict(table, orient='index', columns=land_layers)
lookup.index.name = 'landcover'
lookup.reset_index(inplace=True)
lookup.insert(0, 'class_id', lookup.index)
# Save lookup table to csv
lookup_csv_path = os.path.normpath(os.path.join('test/data/lookup', 'lookup_land.csv'))
lookup.to_csv(lookup_csv_path, index=False)

In [32]:
# Run HydroMT
model_path = os.path.normpath(f'{test_folder}/model')
if os.path.exists(model_path): shutil.rmtree(model_path)
!hydromt build wflow_sbm "./test/model" -i "./test/build.yml" -d "./test/config.yml" -v

2026-05-18 19:09:19,056 - hydromt - log - INFO - HydroMT version: 1.3.1
2026-05-18 19:09:19,102 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from ./test/config.yml
2026-05-18 19:09:19,121 - hydromt.model.model - model - INFO - Initializing wflow_sbm model from hydromt_wflow (v1.0.1).
2026-05-18 19:09:19,121 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from C:\Envs\hyd_ai\Lib\site-packages\hydromt_wflow\data\parameters_data.yml
2026-05-18 19:09:19,140 - hydromt.hydromt_wflow.wflow_base - wflow_base - INFO - Supported Wflow.jl version v1+
2026-05-18 19:09:19,140 - hydromt.hydromt_wflow.components.config - config - INFO - Reading default config file from C:/Envs/hyd_ai/Lib/site-packages/hydromt_wflow/data/wflow_sbm/wflow_sbm.toml.
2026-05-18 19:09:19,142 - hydromt - log - INFO - HydroMT version: 1.3.1
2026-05-18 19:09:19,142 - hydromt.model.model - model - INFO - build: setup_config
2026-05-18 19:09:19,142 - hydromt.m

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Envs\hyd_ai\Scripts\hydromt.exe\__main__.py", line 5, in <module>
  File "C:\Envs\hyd_ai\Lib\site-packages\click\core.py", line 1514, in __call__
    return self.main(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Envs\hyd_ai\Lib\site-packages\click\core.py", line 1435, in main
    rv = self.invoke(ctx)
         ^^^^^^^^^^^^^^^^
  File "C:\Envs\hyd_ai\Lib\site-packages\click\core.py", line 1902, in invoke
    return _process_result(sub_ctx.command.invoke(sub_ctx))
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Envs\hyd_ai\Lib\site-packages\click\core.py", line 1298, in invoke
    return ctx.invoke(self.callback, **ctx.params)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Envs\hyd_ai\Lib\site-packages\click\core.py", line 853, in invoke
    return callback(*args, **kwargs)
        